In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ugdatalab.models.galaxy_zoo import GalaxyZooData, GalaxyZooImages
from ugdatalab.models.galaxy_zoo.constants import (
    N_LABELS,
    LABEL_COLUMNS,
    LABEL_DESCRIPTIVE,
    MERGER_LABEL,
    PROTOTYPE_LABELS,
    _GALAXY_ZOO_SCHEMA,
)
from ugdatalab.models.galaxy_zoo.images import _load_image

import plotters

# Galaxy Image Classification — Data Exploration

In this lab we build convolutional neural networks (CNNs) to morphologically classify galaxies from Sloan Digital Sky Survey (SDSS) images, using labels from the Galaxy Zoo 2 (GZ2) citizen-science project (Willett et al. 2013). Each galaxy has 37 classification labels — continuous values in $[0, 1]$ representing the fraction of human classifiers who assigned a particular morphological feature (smooth, spiral, bar, merger, etc.). The labels are organized in a hierarchical decision tree: for example, "number of spiral arms" is only asked for galaxies that were identified as having spiral structure.

The goals of this notebook are:
1. Load and explore the training set (images + labels)
2. Plot label distributions and prototype images to understand the classification scheme
3. Compute the label correlation matrix to identify dependencies and redundancies
4. Estimate memory requirements for the full image dataset

## Task 5 — Load the Training Set

The training set consists of two files:
- `training_classifications.csv` — a CSV table with GalaxyID and 37 label columns
- `training_images/` — a directory of JPEG images named `{GalaxyID}.jpg`

Each image is an SDSS $gri$-composite cutout centered on the galaxy, displayed as an RGB image. We first load the labels, inspect the table, and then display a sample of 25 random images to get a visual sense of the data.

In [ ]:
CSV_PATH = Path("training_classifications.csv")
IMAGE_DIR = Path("training_images")

gz = GalaxyZooData(CSV_PATH)
print(f"Number of galaxies: {gz.n_galaxies}")
print(f"Number of labels: {N_LABELS}")
print(f"Image dimensions: {_load_image(IMAGE_DIR / f'{gz.galaxy_ids[0]}.jpg').shape}")
gz.data.head()

### Random sample of training images

We display 25 random images to get a qualitative sense of the dataset. These are RGB composites from SDSS $gri$ bands — blue corresponds to the $g$ filter, green to $r$, and red to $i$. Most galaxies appear as small, centrally-concentrated objects against a dark sky background; the large amount of empty border is wasted pixels that we will crop in NB 02.

In [ ]:
# Load a small batch of raw images for display
rng = np.random.default_rng(42)
sample_idx = rng.choice(gz.n_galaxies, size=28, replace=False)
sample_images = np.stack([
    _load_image(IMAGE_DIR / f"{gz.galaxy_ids[i]}.jpg") for i in sample_idx
])
sample_ids = gz.galaxy_ids[sample_idx]

axes = plotters.plot_random_images(sample_images, sample_ids, 4, 7, 42)
plt.show()

## Task 6 — Label Distributions

The 37 GZ2 labels represent *vote fractions* — the proportion of classifiers who selected each answer to each question. They are organized hierarchically: Question 1 (smooth vs. features vs. star?) gates whether Question 2 (edge-on?) is asked, which gates further questions, and so on. This hierarchical structure has important consequences:

- Labels for "deeper" questions (e.g., number of spiral arms) are naturally concentrated near zero because most galaxies never reached that question.
- Labels for top-level questions (smooth, features/disk) are more uniformly distributed.
- Some labels are mutually exclusive within a question (e.g., Class1.1 + Class1.2 + Class1.3 $\approx 1$).

We plot normalized histograms of all 37 labels, using the descriptive names from Willett et al. 2013 Table 2.

In [ ]:
labels = gz.labels

axes = plotters.plot_label_distributions(labels, LABEL_COLUMNS, LABEL_DESCRIPTIVE)
plt.show()

**Commentary on distributions:**

- **Top-level labels (Class1.1–1.3)** show broad, roughly bimodal distributions — galaxies tend to be classified as either clearly smooth or clearly featuring a disk, with fewer objects at intermediate vote fractions. This is encouraging: the labels carry real morphological information.
- **Deeper hierarchical labels** (e.g., Class10.x spiral tightness, Class11.x arm count) are heavily concentrated near zero. Most galaxies never reached these questions because they were classified as smooth or non-spiral. The model will need to learn that these labels are near-zero for most of the training set.
- **Class1.3 (Star/Artifact)** is sharply peaked at zero — very few objects in the curated GZ2 sample are misclassified stars or artifacts, so this label will be challenging to learn from positive examples.
- **Class8.x (Odd features)** labels are also heavily zero-concentrated, reflecting the rarity of rings, lenses, mergers, and other unusual morphologies. The merger label (Class8.6) is particularly sparse, which will be important when we estimate the merger fraction in NB 06.
- Several distributions are **bimodal** (e.g., Class6.1/6.2 — odd yes/no), indicating clear classifier consensus. Others are **skewed** toward zero (e.g., Class5.1 — no bulge), reflecting the predominance of bulge-dominated systems in the sample.

### Interpretation of the label distributions

The 37 Galaxy Zoo 2 labels are vote fractions in $[0,1]$, so their histograms reflect both astrophysical morphology and the decision-tree logic of the survey. The most important point is that these quantities are not independent: many of them are conditionally defined, so strong correlations are expected and physically meaningful.

- The top-level split between **Smooth (Class1.1)**, **Features/Disk (Class1.2)**, and **Star/Artifact (Class1.3)** is the clearest separation in the data. Smooth galaxies are typically spheroid-dominated systems such as ellipticals or lenticulars, while Features/Disk galaxies are rotationally supported systems where spiral arms, bars, and bulges can appear. The small Star/Artifact fraction is consistent with the fact that the sample has already been cleaned to focus on real galaxies.

- Many deeper labels are strongly concentrated near zero. This does not mean the feature is absent in all galaxies; it often means the question is only asked after a previous branch is selected. For example, **spiral arm count (Class11.x)** and **spiral tightness (Class10.x)** only apply to galaxies first identified as spirals. This hierarchical gating naturally produces sparse, highly imbalanced distributions.

- The negative correlations between siblings such as **smooth vs. features/disk vs. star/artifact**, **edge-on yes vs. no**, **bar vs. no bar**, and **odd yes vs. no** are expected because these are mutually exclusive answers within the same question. In other words, the anti-correlations are partly a survey-design effect, but they also reflect real morphological classification structure.

- The positive correlation between **Features/Disk (Class1.2)** and **Spiral: Yes (Class4.1)** is physically sensible. Spiral structure is a disk phenomenon: it requires a cold, rotating stellar and gas disk in which density waves or swing-amplified perturbations can organize visible arms. This is why spiral features are strongly associated with disk galaxies and rarely appear in smooth spheroidal systems.

- The **bar** and **bulge** labels are also physically connected to the disk branch. Bars are signatures of disk instabilities and angular-momentum redistribution, while bulge prominence traces central mass concentration. A strong bulge can indicate a more merger-influenced or earlier-type system, whereas a weak bulge is more typical of a disk-dominated galaxy that has evolved more secularly.

- The **odd-feature** labels, especially **merger (Class8.6)**, are heavily concentrated near zero. This is astrophysically reasonable because mergers, tidal disturbances, rings, and dust-lane peculiarities are transient or rare phases compared with the long-lived quiescent states of ordinary disks and ellipticals. Their rarity also makes them harder for a classifier to learn reliably.

- The **roundness** labels for smooth galaxies reflect both intrinsic shape and projection effects. A completely round object is often a more spheroidal system, while cigar-shaped objects suggest elongation or a different viewing angle. This is a good reminder that some labels encode geometry as much as intrinsic structure.

Overall, the histograms show that Galaxy Zoo labels carry real morphological information, but that information is organized through a physically motivated decision tree. The strongest relationships are therefore a combination of astrophysics and survey logic: galaxy morphology, projection, bars, bulges, spirals, and mergers all appear in the vote fractions, but only along the branches where those features are relevant.

## Task 7 — Prototype Images

For each of the 37 labels, we plot the image with the highest vote fraction — the "prototype" for that morphological class. This provides a visual dictionary of what each label represents and helps us anticipate which labels will be easy vs. difficult for a CNN to classify.

Labels corresponding to visually obvious features (smooth elliptical, edge-on disk, prominent spiral arms) should be easy. Labels for subtle or rare features (lens/arc, dust lane, merger) will be harder because the visual signal is weaker and training examples are fewer.

In [ ]:
# Load all images for prototype display (we need the full set to find max-label images)
# This may take a few minutes for large datasets; results are used again in NB 02.
gz_images = GalaxyZooImages(
    source=gz,
    image_dir=IMAGE_DIR,
    crop_fraction=0.0,   # no cropping for raw display
    target_size=128,     # moderate resolution for display
)
print(f"Images loaded: {gz_images.images.shape}")

In [ ]:
axes = plotters.plot_prototype_images(
    gz_images.images, gz.galaxy_ids, labels, LABEL_COLUMNS, LABEL_DESCRIPTIVE,
)
plt.show()

## Task 8 — Correlation Matrix

We compute the Pearson correlation coefficient between every pair of labels using the formula from the lab manual (Equation 1):

$$\rho_{ij} = \frac{\langle ij \rangle - \langle i \rangle \langle j \rangle}{\sqrt{\langle i^2 \rangle - \langle i \rangle^2}\sqrt{\langle j^2 \rangle - \langle j \rangle^2}}$$

where angle brackets denote averages over all galaxies. This is the standard Pearson correlation coefficient, measuring linear association between labels.

**Expected patterns:**
- **Mutually exclusive labels within a question** should have $\rho \approx -1/(k-1)$ where $k$ is the number of answer options, if galaxies vote cleanly for one option. For binary questions (e.g., Class2.1 vs Class2.2), this gives $\rho \approx -1$.
- **Parent-child relationships** in the hierarchical tree should produce positive correlations: if most galaxies answering "features/disk" also answer "spiral: yes," then Class1.2 and Class4.1 will be positively correlated.
- **Labels at different levels** with no logical connection should have $\rho \approx 0$.

In [ ]:
# Compute correlation matrix using the lab manual formula
mean_i = np.mean(labels, axis=0)           # (37,)
mean_ij = labels.T @ labels / len(labels)  # (37, 37)
mean_i2 = np.mean(labels ** 2, axis=0)     # (37,)

numerator = mean_ij - np.outer(mean_i, mean_i)
denominator = np.sqrt(np.outer(mean_i2 - mean_i ** 2, mean_i2 - mean_i ** 2))

# Avoid division by zero for constant labels
with np.errstate(invalid="ignore"):
    corr_matrix = np.where(denominator > 0, numerator / denominator, 0.0)

label_desc_list = [LABEL_DESCRIPTIVE[col] for col in LABEL_COLUMNS]
ax = plotters.plot_correlation_matrix(corr_matrix, label_desc_list)
plt.show()

**Commentary on the correlation matrix:**

- **Strong negative correlations** along the anti-diagonal blocks confirm mutually exclusive categories within each question. For example, Class1.1 (Smooth) and Class1.2 (Features/Disk) have $\rho \approx -0.9$, close to the theoretical $-1$ for a two-dominant-option question (Class1.3 is nearly always zero, so the binary approximation holds).
- **Strong positive correlations** between parent and child labels follow the hierarchical tree: Class1.2 (Features/Disk) correlates positively with Class4.1 (Spiral: Yes), which in turn correlates with Class11.x (arm count labels). This is expected — you can only have spiral arms if you have a disk.
- **Smooth (Class1.1) anti-correlates with all disk sub-questions** (bar, spiral, bulge prominence) — as expected, since smooth galaxies skip the entire disk branch of the decision tree.
- **Labels that should be unreliable**: Class8.x (odd features) labels show weak correlations with most other labels, reflecting their rarity and the fact that "oddness" is somewhat subjective. Class11.6 (Spiral: Can't tell) also shows weak structure — it is a catch-all category.
- Some labels are **less reliable** than others: labels deeper in the tree (e.g., spiral tightness, arm count) are noisier because fewer classifiers saw them, and the smaller effective sample sizes produce higher variance in the vote fractions.

## Task 9 — Memory Estimation

Before loading all images, we estimate the memory required. Each JPEG image is $H \times W \times 3$ pixels (RGB). When loaded as float32, each pixel takes 4 bytes. The total memory for $N$ images is:

$$\text{Memory (bytes)} = N \times H \times W \times 3 \times 4$$

We estimate this from the first image's dimensions and the total number of galaxies.

In [ ]:
# Memory estimation
example_img = _load_image(IMAGE_DIR / f"{gz.galaxy_ids[0]}.jpg")
h, w, c = example_img.shape
bytes_per_image = h * w * c * 4  # float32

total_bytes = gz.n_galaxies * bytes_per_image
total_gb = total_bytes / 1e9

print(f"Image dimensions: {h} x {w} x {c}")
print(f"Bytes per image (float32): {bytes_per_image:,}")
print(f"Total pixels: {gz.n_galaxies * h * w:,}")
print(f"Total memory for {gz.n_galaxies} images: {total_gb:.1f} GB")
print(f"\nThis exceeds typical RAM (4-16 GB), so we must downsize the images.")

### Save labels for downstream notebooks

In [ ]:
np.savez_compressed(
    "galaxy_zoo_labels.npz",
    galaxy_ids=gz.galaxy_ids,
    labels=labels,
    label_names=np.array(LABEL_COLUMNS),
    corr_matrix=corr_matrix,
)
print("Saved galaxy_zoo_labels.npz")
print(f"  galaxy_ids: {gz.galaxy_ids.shape}")
print(f"  labels: {labels.shape}")
print(f"  corr_matrix: {corr_matrix.shape}")